# Task 3.3 — Trích xuất Code Property Graph cho 9.077 DiverseVul Functions trên Kaggle

Notebook này thiết lập môi trường trích xuất **Graph A** (AST) và **Graph B** (AST + CFG + DATA_DEP) sử dụng công cụ **Joern v4.0.654** trên môi trường điện toán đám mây Kaggle (Chế độ **CPU: 4 vCPU, 30 GB RAM**, hoàn toàn không tiêu tốn quota GPU).

### Quy ước phân chia Batch (Chia 3 phần chạy độc lập hoặc song song):
- **Batch 1**: Mẫu index `0` $\rightarrow$ `3.000` (3.000 hàm)
- **Batch 2**: Mẫu index `3.000` $\rightarrow$ `6.000` (3.000 hàm)
- **Batch 3**: Mẫu index `6.000` $\rightarrow$ `9.077` (3.077 hàm)

> **Lưu ý quan trọng**:
> - Bật **Internet = On** ở thanh cài đặt bên phải notebook.
> - Chọn **Accelerator = None** (CPU mode).
> - Khi chạy Background: Bấm **Save Version** $\rightarrow$ Chọn **Save & Run All (Commit)** để notebook tự chạy ngầm tối đa 12 giờ.

## 1. Cài đặt Môi trường (OpenJDK 21, Joern v4.0.654) & Clone Repository

In [ ]:
import os
import sys
import subprocess

print("=== [1/3] Cài đặt OpenJDK 21 ===")
!apt-get update -qq && apt-get install -y -qq openjdk-21-jdk

print("=== [2/3] Cài đặt Joern v4.0.654 ===")
!curl -sSL https://github.com/joernio/joern/releases/download/v4.0.654/joern-install.sh | bash

# Thiết lập biến môi trường
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = "/root/bin:" + os.environ["PATH"]

print("=== Kiểm tra phiên bản công cụ ===")
!java -version
!joern --version

print("=== [3/3] Clone mã nguồn nhánh feature/3_3_graph-extract ===")
!rm -rf /kaggle/working/repo
!git clone -b feature/3_3_graph-extract https://github.com/meo225/software-bug-detection-using-graphs.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pyyaml

## 2. Kiểm tra Dữ liệu Đầu vào (Dataset Gốc & Manifest Thí nghiệm)

In [ ]:
from pathlib import Path

# Đường dẫn file dataset DiverseVul gốc tải lên từ Kaggle Dataset
# Lưu ý: Thay đổi tên dataset input tương ứng nếu tên dataset trên Kaggle của bạn khác
DATASET_CANDIDATES = [
    Path("/kaggle/input/diversevul-raw/diversevul.jsonl"),
    Path("/kaggle/input/diversevul-dataset/diversevul.jsonl"),
    Path("/kaggle/input/diversevul/diversevul.jsonl"),
]

DATASET_PATH = None
for p in DATASET_CANDIDATES:
    if p.is_file():
        DATASET_PATH = p
        break

MANIFEST_PATH = Path("data/splits/diversevul_experiment_manifest.csv")

if DATASET_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy diversevul.jsonl trong /kaggle/input/! "
        "Vui lòng nạp dataset diversevul.jsonl vào notebook qua nút '+ Add Input'."
    )

assert MANIFEST_PATH.is_file(), f"Không tìm thấy manifest tại {MANIFEST_PATH}"

print(f"✓ Dataset gốc: {DATASET_PATH} ({DATASET_PATH.stat().st_size / (1024**2):.1f} MB)")
print(f"✓ Manifest thí nghiệm: {MANIFEST_PATH}")

## 3. Thực thi Trích xuất Graph A & B theo Batch

Bỏ comment **đúng 1 dòng** tương ứng với Batch bạn muốn chạy trên notebook này (giữ nguyên comment ở 2 dòng còn lại).

- Mỗi batch xử lý độc lập khoảng 3.000 hàm trên 4 workers song song.
- Script tự động checkpoint và resume nếu bị gián đoạn.

In [ ]:
# =========================================================================
# CHỌN BATCH BẠN MUỐN CHẠY (BỎ COMMENT 1 DÒNG DUY NHẤT, COMMENT 2 DÒNG CÒN LẠI)
# =========================================================================

# DÒNG 1: Chạy Batch 1 (Mẫu 0 -> 3.000)
!python scripts/extract_subset.py --dataset {DATASET_PATH} --manifest data/splits/diversevul_experiment_manifest.csv --output-dir /kaggle/working/output_batch_1 --batch 1 --workers 4

# DÒNG 2: Chạy Batch 2 (Mẫu 3.000 -> 6.000)
# !python scripts/extract_subset.py --dataset {DATASET_PATH} --manifest data/splits/diversevul_experiment_manifest.csv --output-dir /kaggle/working/output_batch_2 --batch 2 --workers 4

# DÒNG 3: Chạy Batch 3 (Mẫu 6.000 -> 9.077)
# !python scripts/extract_subset.py --dataset {DATASET_PATH} --manifest data/splits/diversevul_experiment_manifest.csv --output-dir /kaggle/working/output_batch_3 --batch 3 --workers 4

## 4. Đóng gói Artifacts thành File ZIP để Tải về

Nén thư mục kết quả của batch vừa hoàn tất thành file `.zip` trong `/kaggle/working/` để dễ dàng tải về máy local.

In [ ]:
import glob
import os

output_dirs = sorted(glob.glob("/kaggle/working/output_batch_*"))
for out_dir in output_dirs:
    batch_name = os.path.basename(out_dir)
    zip_path = f"/kaggle/working/{batch_name}.zip"
    print(f">>> Đang nén {out_dir} thành {zip_path}...")
    !zip -q -r {zip_path} {out_dir}
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"✓ Hoàn tất đóng gói: {zip_path} ({size_mb:.2f} MB)")

print("\n>>> Bạn có thể tải file .zip này về máy từ tab Output bên phải màn hình Kaggle.")